In [2]:
import os
import numpy as np
import pandas as pd

BASE_PATH = "../../../data"
SPLITS = ["training", "test", "final test"]
FEATURES = ["Temperature", "Pressure", "RPM", "Vibration"]
FEATURE_INDEX = {name: i for i, name in enumerate(FEATURES)}
SEED = 42
rng = np.random.default_rng(SEED)

# =====================================================
# CHANGE (v2.1): two-tier uncalibrated design.
# The rule-based detector only catches EXTREME out-of-range values.
# M0 is now trained to ALSO catch the MODERATE zone -- values above
# any valid operating range but BELOW the rule thresholds. This zone
# is invisible to the rules; only M0 can flag it at runtime.
#
#   Variable     valid (max of generators)  MODERATE (M0 only)          EXTREME (rules)
#   Temperature  -20 .. 145                 146.0..164.9 / -29.9..-21.0  165..300 / -90..-30
#   Pressure     0.58 .. 1.16 (sim loads)   1.30..1.45  /  0.05..0.45    1.5..6  / -6..0.01
#   RPM          0 .. 9000                  9500..11900                  12000..100000 / negative
#   Vibration    ~0 .. 0.6                  0.80..1.90                   2..20 / negative
#
# MODERATE_PROB = 0.5 -> the training set is 50/50 moderate/extreme,
# so M0 learns both frontiers with equal weight.
# =====================================================
MODERATE_PROB = 0.5

# ----------------- ERROR FUNCS (feature-wide) -----------------
def _sample_moderate(feature_name: str, size: int):
    """Uncalibrated values in the MODERATE zone: outside valid ranges,
    below the rule-detector thresholds. Only M0 can catch these."""
    if feature_name == "Temperature":
        return (rng.uniform(-29.9, -21.0, size=size) if rng.random() < 0.5
                else rng.uniform(146.0, 164.9, size=size))
    elif feature_name == "Pressure":
        return (rng.uniform(0.05, 0.45, size=size) if rng.random() < 0.5
                else rng.uniform(1.30, 1.45, size=size))
    elif feature_name == "RPM":
        return rng.uniform(9500.0, 11900.0, size=size)
    else:  # Vibration
        return rng.uniform(0.80, 1.90, size=size)


def _sample_extreme(feature_name: str, size: int):
    """Uncalibrated values in the EXTREME zone (same ranges the
    rule-based detector catches). Unchanged from the original."""
    if feature_name == "Temperature":
        return (rng.uniform(-90.0, -30.0, size=size) if rng.random() < 0.5
                else rng.uniform(165.0, 300.0, size=size))
    elif feature_name == "Pressure":
        return (rng.uniform(-6.0, 0.01, size=size) if rng.random() < 0.5
                else rng.uniform(1.5, 6.0, size=size))
    elif feature_name == "RPM":
        return (rng.uniform(-2000.0, -0.1, size=size) if rng.random() < 0.5
                else rng.uniform(12000.0, 100000.0, size=size))
    else:  # Vibration
        return (rng.uniform(-20.0, -0.1, size=size) if rng.random() < 0.5
                else rng.uniform(2.0, 20.0, size=size))


def _sample_out_of_range(feature_name: str, size: int):
    """Pick moderate or extreme zone per corrupted column."""
    if rng.random() < MODERATE_PROB:
        return _sample_moderate(feature_name, size)
    return _sample_extreme(feature_name, size)

def error_func_out_of_range(col4: np.ndarray, feature_name: str) -> np.ndarray:
    """Corrupt the entire 4-timestep column with out-of-range 'uncalibrated' values."""
    return _sample_out_of_range(feature_name, size=col4.shape[0]).astype(col4.dtype)

# Only keep out_of_range (no hard-coded sentinel error_value anymore)
ERROR_FUNCS = {
    "out_of_range": error_func_out_of_range,
}
ERROR_NAMES = list(ERROR_FUNCS.keys())
# With a single error type, prob vector is trivial (kept for future extensibility)
ERROR_PROBS = np.array([1.0], dtype=float)

# ----------------- OCC APPLICATION -----------------
def apply_occ_random(window_4x4: np.ndarray,
                     rng: np.random.Generator,
                     force_error: str | None = None) -> np.ndarray:
    """
    window_4x4: (4,4) [Time x Features].
    force_error: None -> choose from ERROR_FUNCS (here only 'out_of_range');
                 or 'out_of_range' explicitly.
    """
    w = window_4x4.astype(np.float32).copy()

    # how many variables get errors (1..4)
    k_vars = int(rng.integers(1, len(FEATURES) + 1))
    var_indices = rng.choice(len(FEATURES), size=k_vars, replace=False)

    for vi in var_indices:
        feat = FEATURES[vi]
        if force_error is None:
            chosen_name = rng.choice(ERROR_NAMES, p=ERROR_PROBS)  # effectively always 'out_of_range'
            err_fn = ERROR_FUNCS[chosen_name]
        else:
            err_fn = ERROR_FUNCS[force_error]
        w[:, vi] = err_fn(w[:, vi], feat)

    return w

# ----------------- I/O -----------------
def load_router(split: str):
    split_dir = os.path.join(BASE_PATH, split)
    X = np.load(os.path.join(split_dir, f"Router_{split}_X.npy"))  # (N,4,4)
    y = np.load(os.path.join(split_dir, f"Router_{split}_y.npy"))  # (N,)
    return X, y, split_dir

def save_occ(split_dir: str, split: str, X_occ: np.ndarray, y_occ: np.ndarray):
    np.save(os.path.join(split_dir, f"RouterOCC_{split}_X.npy"), X_occ)
    np.save(os.path.join(split_dir, f"RouterOCC_{split}_y.npy"), y_occ)

    rows = []
    N = X_occ.shape[0]
    for seq in range(N):
        for t in range(4):
            rows.append({
                "Time": t + 1,
                "Sequence": seq + 1,
                "Temperature": float(X_occ[seq, t, FEATURE_INDEX["Temperature"]]),
                "Pressure":    float(X_occ[seq, t, FEATURE_INDEX["Pressure"]]),
                "RPM":         float(X_occ[seq, t, FEATURE_INDEX["RPM"]]),
                "Vibration":   float(X_occ[seq, t, FEATURE_INDEX["Vibration"]]),
                "State":       3
            })
    df = pd.DataFrame(rows, columns=["Time","Sequence",*FEATURES,"State"])
    df.to_csv(os.path.join(split_dir, f"RouterOCC_{split}.csv"), index=False)

# ----------------- MAIN -----------------
# Variants included: "mixed" (which, with a single error type, equals out_of_range) + per-error version.
ERROR_VARIANTS = ["mixed", "out_of_range"]

def process_split(split: str):
    X_router, _, split_dir = load_router(split)
    N = X_router.shape[0]

    parts = []
    for variant in ERROR_VARIANTS:
        X_occ = np.empty_like(X_router, dtype=np.float32)
        if variant == "mixed":
            for i in range(N):
                X_occ[i] = apply_occ_random(X_router[i], rng, force_error=None)  # effectively out_of_range
        else:  # 'out_of_range'
            for i in range(N):
                X_occ[i] = apply_occ_random(X_router[i], rng, force_error="out_of_range")
        parts.append(X_occ)

    X_all = np.concatenate(parts, axis=0)
    y_all = np.full((X_all.shape[0],), 3, dtype=np.int32)

    save_occ(split_dir, split, X_all, y_all)
    return X_all.shape, y_all.shape

for s in SPLITS:
    process_split(s)

print("✅ Router OCC datasets created (moderate + extreme uncalibrated, 50/50).")

✅ Router OCC datasets created (moderate + extreme uncalibrated, 50/50).
